# YOLOv8 Standalone Flag Classification Model Pipeline

This notebook runs the **entire flag classification pipeline** entirely on Kaggle:
1. Installs dependencies.
2. Optionally generates a synthetic dataset from template flags and background negatives.
3. Optionally loads and splits the public 296-class cropped flag dataset.
4. Trains a `yolov8s-cls` image classifier.
5. Runs local custom validation accuracy check.
6. Packs weights and plots for download.

## Step 1: Install Dependencies

In [ ]:
!pip install -U ultralytics

## Step 2 (Option A): Generate Synthetic Flag Dataset
Run this cell if you want to generate a synthetic classification dataset from template flags and background negative images.

In [ ]:
import os
import glob
import shutil
import random
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

def clean_dir(path):
    import time
    for _ in range(5):
        try:
            if os.path.exists(path):
                shutil.rmtree(path)
            os.makedirs(path, exist_ok=True)
            return
        except Exception:
            time.sleep(0.5)
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def main():
    base_dir = r"C:\Users\mido\Documents\antigravity\focused-babbage"
    dataset_src = os.path.join(base_dir, "mar_258class_dataset", "final_dataset", "Flag_Training_Project")
    yaml_path = os.path.join(dataset_src, "dataset.yaml")
    dst_dir = os.path.join(base_dir, "kaggle_classification_dataset")
    full_neg_dir = os.path.join(base_dir, "full_frame_negatives")
    ref_country_dir = os.path.join(base_dir, "reference", "country")
    ref_inst_dir = os.path.join(base_dir, "reference", "institution")

    # 1. Parse dataset.yaml to get class names
    if not os.path.exists(yaml_path):
        print(f"[ERROR] dataset.yaml not found at {yaml_path}")
        return

    names = {}
    with open(yaml_path, 'r') as f:
        lines = f.readlines()
    
    in_names = False
    for line in lines:
        if line.strip().startswith("names:"):
            in_names = True
            continue
        if in_names:
            if line.strip() == "":
                continue
            parts = line.split(":")
            if len(parts) >= 2:
                try:
                    cls_id = int(parts[0].strip())
                    cls_name = parts[1].strip().strip("'\"")
                    names[cls_id] = cls_name
                except ValueError:
                    continue
    print(f"Loaded {len(names)} class names from dataset.yaml.")

    # 2. Map class names to template paths
    template_paths = {}
    for path in glob.glob(os.path.join(ref_country_dir, "*")):
        if path.lower().endswith(".png"):
            name = os.path.splitext(os.path.basename(path))[0]
            template_paths[name] = path
    for path in glob.glob(os.path.join(ref_inst_dir, "*")):
        if path.lower().endswith((".png", ".jpg", ".jpeg", ".gif")):
            name = os.path.splitext(os.path.basename(path))[0]
            template_paths[name] = path

    class_to_template = {}
    missing_templates = []
    for cls_id, cls_name in names.items():
        if cls_name in template_paths:
            class_to_template[cls_name] = template_paths[cls_name]
        else:
            missing_templates.append(cls_name)

    if missing_templates:
        print(f"[WARNING] Missing template files for classes: {missing_templates}")
    else:
        print("All templates successfully mapped to class names.")

    # 3. Load negative background images
    # We collect all negative images from the original dataset and full_frame_negatives
    dataset_neg_files = []
    all_jpgs = glob.glob(os.path.join(dataset_src, "*.jpg"))
    for img_path in all_jpgs:
        txt_path = os.path.splitext(img_path)[0] + ".txt"
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                lines = f.readlines()
            if not lines or not any(l.strip() for l in lines):
                dataset_neg_files.append(img_path)
        else:
            dataset_neg_files.append(img_path)

    all_negatives = list(dataset_neg_files)
    if os.path.exists(full_neg_dir):
        neg_files_extra = glob.glob(os.path.join(full_neg_dir, "*.jpg"))
        all_negatives.extend(neg_files_extra)
        print(f"Loaded {len(neg_files_extra)} extra negatives.")

    print(f"Total negative images loaded: {len(all_negatives)}")
    if not all_negatives:
        print("[ERROR] No background negative images found!")
        return

    # 4. Initialize directories
    print("\n=== Initializing classification dataset folders ===")
    for split in ['train', 'val']:
        clean_dir(os.path.join(dst_dir, split))
        for class_name in names.values():
            os.makedirs(os.path.join(dst_dir, split, class_name), exist_ok=True)
        os.makedirs(os.path.join(dst_dir, split, "background"), exist_ok=True)

    # 5. Helper function to generate a single crop
    def generate_synthetic_crop(template_path, bg_images, size=128):
        # 1. Crop background patch
        bg_img = None
        for _ in range(10):
            try:
                bg_path = random.choice(bg_images)
                with Image.open(bg_path) as bi:
                    bw_img, bh_img = bi.size
                    if bw_img >= size and bh_img >= size:
                        x0 = random.randint(0, bw_img - size)
                        y0 = random.randint(0, bh_img - size)
                        bg_img = bi.crop((x0, y0, x0 + size, y0 + size)).copy()
                        break
            except Exception:
                continue
        if bg_img is None:
            # Fallback background
            bg_img = Image.new('RGB', (size, size), (218, 187, 156))

        # 2. Load and augment flag
        try:
            with Image.open(template_path) as flag_img:
                flag = flag_img.convert('RGBA')
                orig_w, orig_h = flag.size

                # Scale flag randomly
                tw = random.randint(45, 90)
                th = max(5, int((tw / orig_w) * orig_h))
                flag = flag.resize((tw, th), Image.Resampling.LANCZOS)

                # Rotate flag randomly
                angle = random.uniform(-20, 20)
                flag = flag.rotate(angle, expand=True)

                # Paste flag on background at center + offset
                fw, fh = flag.size
                cx = size // 2 - fw // 2
                cy = size // 2 - fh // 2
                dx = random.randint(-15, 15)
                dy = random.randint(-15, 15)

                bg_img.paste(flag, (cx + dx, cy + dy), flag)
        except Exception as e:
            # If template fails, return background as-is (graceful degradation)
            pass

        # 3. Apply lens effects (brightness, contrast, color, blur)
        bg_img = ImageEnhance.Color(bg_img).enhance(random.uniform(0.85, 1.25))
        bg_img = ImageEnhance.Brightness(bg_img).enhance(random.uniform(0.85, 1.15))
        bg_img = ImageEnhance.Contrast(bg_img).enhance(random.uniform(0.85, 1.20))
        
        if random.random() < 0.30:
            bg_img = bg_img.filter(ImageFilter.GaussianBlur(random.uniform(0.1, 0.4)))
            
        return bg_img.convert('RGB')

    # 6. Generate synthetic images for all 254 classes
    random.seed(42)
    
    num_train_per_class = 60
    num_val_per_class = 15
    
    print("\n=== Generating synthetic flag classification dataset ===")
    for cls_name, t_path in class_to_template.items():
        # Train split
        for i in range(num_train_per_class):
            crop = generate_synthetic_crop(t_path, all_negatives, size=128)
            dst_path = os.path.join(dst_dir, "train", cls_name, f"synth_{cls_name}_{i:03d}.jpg")
            crop.save(dst_path, 'JPEG', quality=95)
            
        # Val split
        for i in range(num_val_per_class):
            crop = generate_synthetic_crop(t_path, all_negatives, size=128)
            dst_path = os.path.join(dst_dir, "val", cls_name, f"synth_{cls_name}_{i:03d}.jpg")
            crop.save(dst_path, 'JPEG', quality=95)

    print(f"Generated {num_train_per_class} train and {num_val_per_class} val crops per class.")

    # 7. Generate background class crops
    print("\n=== Generating background/negative class crops ===")
    num_train_bg = 150
    num_val_bg = 40

    # Train background
    for i in range(num_train_bg):
        crop = generate_synthetic_crop(None, all_negatives, size=128)
        dst_path = os.path.join(dst_dir, "train", "background", f"bg_patch_{i:04d}.jpg")
        crop.save(dst_path, 'JPEG', quality=90)

    # Val background
    for i in range(num_val_bg):
        crop = generate_synthetic_crop(None, all_negatives, size=128)
        dst_path = os.path.join(dst_dir, "val", "background", f"bg_patch_{i:04d}.jpg")
        crop.save(dst_path, 'JPEG', quality=90)

    print(f"Generated {num_train_bg} train background crops and {num_val_bg} val background crops.")
    print(f"\n=== Synthetic classification dataset generation complete! Saved under {dst_dir} ===")

if __name__ == '__main__':
    main()


## Step 2 (Option B): Load & Split Public 296-Class Flag Dataset
Run this cell if you want to use the public 296-class cropped flag dataset.

In [ ]:
import os
import shutil
import random
from pathlib import Path

# Paths
src_dataset = Path("/kaggle/input/new-dataset-updated-296-cropped-mergedd/new-DATASET_updated_296_cropped_Merged")
src_bg_dataset = Path("/kaggle/input/drone-flag-classification-dataset-2026")
dst_dataset = Path("/kaggle/working/dataset")

# Reset destination
if dst_dataset.exists():
    shutil.rmtree(dst_dataset)
dst_dataset.mkdir(parents=True, exist_ok=True)

# Splits
(dst_dataset / "train").mkdir(exist_ok=True)
(dst_dataset / "val").mkdir(exist_ok=True)

# Copy/symlink flag classes
print("Splitting flag classes...")
classes = [c for c in src_dataset.iterdir() if c.is_dir()]
for c in classes:
    class_name = c.name
    # Create train and val class folders
    (dst_dataset / "train" / class_name).mkdir(exist_ok=True)
    (dst_dataset / "val" / class_name).mkdir(exist_ok=True)
    
    # Get all images
    imgs = list(c.glob("*.jpg")) + list(c.glob("*.png")) + list(c.glob("*.jpeg")) + list(c.glob("*.JPG")) + list(c.glob("*.PNG")) + list(c.glob("*.JPEG"))
    random.seed(42)
    random.shuffle(imgs)
    
    # 85/15 split
    split_idx = int(len(imgs) * 0.85)
    train_imgs = imgs[:split_idx]
    val_imgs = imgs[split_idx:]
    
    # Symlink train images
    for idx, img in enumerate(train_imgs):
        os.symlink(img, dst_dataset / "train" / class_name / f"{idx:05d}_{img.name}")
        
    # Symlink val images
    for idx, img in enumerate(val_imgs):
        os.symlink(img, dst_dataset / "val" / class_name / f"{idx:05d}_{img.name}")

# Copy/symlink background class
print("Copying background class...")
(dst_dataset / "train" / "background").mkdir(exist_ok=True)
(dst_dataset / "val" / "background").mkdir(exist_ok=True)

bg_train_src = src_bg_dataset / "train" / "background"
bg_val_src = src_bg_dataset / "val" / "background"

# Symlink train background images
if bg_train_src.exists():
    for img in bg_train_src.glob("*.jpg"):
        os.symlink(img, dst_dataset / "train" / "background" / img.name)
        
# Symlink val background images
if bg_val_src.exists():
    for img in bg_val_src.glob("*.jpg"):
        os.symlink(img, dst_dataset / "val" / "background" / img.name)

print("Dataset preparation complete!")
dataset_dir = "/kaggle/working/dataset"


## Step 3: Initialize Model

In [ ]:
import torch
from ultralytics import YOLO

cuda_available = torch.cuda.is_available()
print(f"GPU (CUDA) Available: {cuda_available}")
if cuda_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    
model = YOLO("yolov8s-cls.pt")
print("Loaded yolov8s-cls.pt base model.")

## Step 4: Train Classification Model

In [ ]:
import os
# Note: set dataset_dir to the split dataset path generated above
try:
    dataset_dir
except NameError:
    # Fallback to synthetic if not explicitly loaded
    dataset_dir = "/kaggle/working/kaggle_classification_dataset"

print(f"Training on dataset directory: {dataset_dir}")
if os.path.exists(dataset_dir):
    device_val = 0 if torch.cuda.is_available() else 'cpu'
    model.train(
        data=dataset_dir,
        epochs=15,
        imgsz=128,
        batch=32,
        device=device_val,
        workers=8,
        name="yolov8s_cls_flag",
        exist_ok=True
    )
else:
    print("Error: Dataset directory not found!")

## Step 5: Verify Classification Accuracy

In [ ]:
import os
import glob
from ultralytics import YOLO

def main():
    base_dir = r"C:\Users\mido\Documents\antigravity\focused-babbage"
    model_path = os.path.join(base_dir, "yolov8s_flag_classification_best.pt")
    val_dir = os.path.join(base_dir, "kaggle_classification_dataset")

    if not os.path.exists(model_path):
        print(f"[ERROR] Classification weights not found at {model_path}")
        return

    if not os.path.exists(os.path.join(val_dir, "val")):
        print(f"[ERROR] Validation dataset folder not found at {os.path.join(val_dir, 'val')}")
        return

    print(f"Loading classification model: {model_path}")
    model = YOLO(model_path)

    print(f"Evaluating classification model on validation split: {os.path.join(val_dir, 'val')}...")
    results = model.val(data=val_dir, split="val", imgsz=128, batch=16, verbose=True)
    
    # YOLO classification validation returns metrics
    top1 = results.top1 * 100
    top5 = results.top5 * 100
    print("\n" + "="*40)
    print(f"Validation Top-1 Accuracy: {top1:.2f}%")
    print(f"Validation Top-5 Accuracy: {top5:.2f}%")
    print("="*40)

if __name__ == '__main__':
    main()


## Step 6: Compress Outputs for Download

In [ ]:
import zipfile

results_dir = "/kaggle/working/runs/classify/yolov8s_cls_flag"
zip_path = "/kaggle/working/yolov8s_cls_flag_results.zip"

if os.path.exists(results_dir):
    print(f"Compressing results from {results_dir}...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(results_dir):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, results_dir))
    print(f"Saved package to: {zip_path}")
else:
    print("Error: Training runs directory not found.")